In [20]:
%load_ext autoreload

%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
from typing import Dict, List, Optional, Tuple, Union, Any, Callable, Mapping
import pandas as pd
import pickle
import numpy as np
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, matthews_corrcoef, balanced_accuracy_score, roc_auc_score, cohen_kappa_score, auc, precision_recall_curve)
from rdkit.Chem import rdFingerprintGenerator
from sklearn.linear_model import LogisticRegression

In [22]:
def fin(df, radius, fpSize):
    fingerprints = []
    onbits_list = []
    fp_generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fpSize)
    for i, mol in enumerate(df["ROMol"]):
        try:
            fp = fp_generator.GetFingerprint(mol)
            # 1になっているビットの位置を取得
            onbits = list(fp.GetOnBits())
            onbits_list.append(onbits)
            
            # NumPy配列も必要なら
            fp_np = fp_generator.GetFingerprintAsNumPy(mol)
            fingerprints.append(fp_np)

        except Exception as e:
            print(f"Error processing molecule {i}: {e}")
            continue
    return np.array(fingerprints), onbits_list

def add_vectors(fp_list: List[List[int]], model: Doc2Vec) -> List[np.ndarray]:
    """Combine document vectors based on fingerprints
    
    Args:
        fp_list: List of fingerprint lists, where each fingerprint is represented as a list of indices
        model: Trained Doc2Vec model containing document vectors
        
    Returns:
        List of compound vectors as numpy arrays
    """
    compound_vec = []
    for i in fp_list:
        fingerprint_vec = 0
        num = 0
        for j in i:
            fingerprint_vec += model.dv.vectors[j]
            num += 1
        compound_vec.append(fingerprint_vec / num if num > 0 else fingerprint_vec)
    return compound_vec

def calculate_metrics(y_true, y_pred, y_proba):

    metrics = {}
    metrics['f1'] = f1_score(y_true, y_pred)
    metrics['mcc'] = matthews_corrcoef(y_true, y_pred)
    metrics['balanced_accuracy'] = balanced_accuracy_score(y_true, y_pred)
    metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    metrics['kappa'] = cohen_kappa_score(y_true, y_pred)
    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    metrics['pr_auc'] = auc(recall, precision)
    return metrics

def evaluate_category(X_train_vec: np.ndarray, 
                      X_test_vec: np.ndarray,
                      y_train: np.ndarray,
                      y_test: np.ndarray,                    
                      lightgbm_model: lgb.LGBMClassifier
                      ) -> Dict[str, Union[List[float], float]]:
    
    # 全ての評価指標のスコアを格納する辞書
    all_train_scores = {'f1': [], 'mcc': [], 'balanced_accuracy': [], 'roc_auc': [], 'kappa': [], 'pr_auc': []}
    all_test_scores = {'f1': [], 'mcc': [], 'balanced_accuracy': [], 'roc_auc': [], 'kappa': [], 'pr_auc': []}

    lightgbm_model.fit(X_train_vec, y_train)
    y_train_pred = lightgbm_model.predict(X_train_vec)
    y_test_pred = lightgbm_model.predict(X_test_vec)
    y_train_proba = lightgbm_model.predict_proba(X_train_vec)[:, 1]
    y_test_proba = lightgbm_model.predict_proba(X_test_vec)[:, 1]

    train_metrics = calculate_metrics(y_train, y_train_pred, y_train_proba)
    test_metrics = calculate_metrics(y_test, y_test_pred, y_test_proba)
    for metric_name in all_train_scores.keys():
        all_train_scores[metric_name].append(train_metrics[metric_name])
        all_test_scores[metric_name].append(test_metrics[metric_name])
    
    # 結果を整理
    results = {}
    for metric_name in all_train_scores.keys():
        results[metric_name] = {
            'train_scores': all_train_scores[metric_name],
            'test_scores': all_test_scores[metric_name],
        }
    
    return results

def make_fp2vector(model_path: str, train_df: pd.DataFrame, test_df: pd.DataFrame) :
    
    model = Doc2Vec.load(model_path)
    train_bit_list = fin(train_df, 3, 8192)[1]
    test_bit_list = fin(test_df, 3, 8192)[1]
    train_compound_vec = add_vectors(train_bit_list, model)
    test_compound_vec = add_vectors(test_bit_list, model)
    X_train_vec = np.array(train_compound_vec)
    X_test_vec = np.array(test_compound_vec)

    return X_train_vec, X_test_vec

def main(train_df: pd.DataFrame, 
         test_df: pd.DataFrame,
         X_train_vec: np.ndarray,
         X_test_vec: np.ndarray,
         lightgbm_model: lgb.LGBMClassifier,) -> Dict[str, Dict[str, float]]:
        
    # Define categories to evaluate
    categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
    
    # Evaluate each category
    results = {}
    for category in categories:
        y_train = np.array([1 if i == category else 0 for i in train_df[category]])
        y_test = np.array([1 if i == category else 0 for i in test_df[category]])
        results[category] = evaluate_category(X_train_vec, X_test_vec, y_train, y_test, lightgbm_model)

    return results

def build_fpdoc2vec_model(corpus: List[List[str]], 
                        tag_list: List[List[int]], #変更
                        doc2vec_param: Dict[str, Any]) -> Doc2Vec:
    """
    Build and train a Doc2Vec model from corpus and structure information
    
    Args:
        corpus: List of lists containing tokenized text for each document
        list: List of lists containing tags for each document
        doc2vec_param: Dictionary of parameters for the Doc2Vec model
        
    Returns:
        Trained Doc2Vec model
    """
    tagged_documents = [
        TaggedDocument(words=corpus, tags=tag_list[i]) #変更
        for i, corpus in enumerate(corpus)
    ]
    
    model = Doc2Vec(tagged_documents, **doc2vec_param)
    
    return model

def make_doc2vector(df: pd.DataFrame, purpose_description: str, tag_list: List[List[int]], doc2vec_param: Dict[str, Any]) -> np.ndarray:

    corpus = df[purpose_description].tolist()#変更
    model = build_fpdoc2vec_model(corpus, tag_list, doc2vec_param)
    return model

!!! Fp doc2vec !!!

In [26]:
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

gbm_params: Dict[str, Any] = {
    "boosting_type": "dart",
    "num_leaves": 48,
    "max_depth": 5,
    "learning_rate": 0.04166324251391809,
    "n_estimators": 736,
    "class_weight": "balanced",
    "min_split_gain": 0.009346925180781129,
    "min_child_weight": 0.0007929549087822909,
    "min_child_samples": 37,
    "reg_alpha": 1.757104268180148,
    "reg_lambda": 1.463369722508726,
    "feature_fraction": 0.50163362868711,
    "feature_fraction_bynode": 0.8321043377994284,
    "subsample": 0.6974385909748512,
    "colsample_bytree": 0.6568268046410831,
    "subsample_freq": 5,
    "drop_rate": 0.24668126335938073,
    "max_drop": 28,
    "skip_drop": 0.5591506516119614,
    "uniform_drop": True,
    "xgboost_dart_mode": True,
    "objective": "binary",
    "random_state": 0,
    "verbose": -1,
    "force_col_wise": True
}

# Create classifier
lightgbm_model = lgb.LGBMClassifier(**gbm_params)

# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    train_df = pickle.load(f)

input_path = "data/test_df2.pkl"
with open(input_path, "rb") as f:
    test_df = pickle.load(f)

# FP Doc2Vec approach
X_train_vec, X_test_vec = make_fp2vector("train_df_doc2vec_8192.model", train_df, test_df)
fpdoc8192_results = main(train_df, test_df, X_train_vec, X_test_vec, lightgbm_model)


In [59]:
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

gbm_params: Dict[str, Any] = {
    "boosting_type": "dart",
    "num_leaves": 48,
    "max_depth": 5,
    "learning_rate": 0.04166324251391809,
    "n_estimators": 736,
    "class_weight": "balanced",
    "min_split_gain": 0.009346925180781129,
    "min_child_weight": 0.0007929549087822909,
    "min_child_samples": 37,
    "reg_alpha": 1.757104268180148,
    "reg_lambda": 1.463369722508726,
    "feature_fraction": 0.50163362868711,
    "feature_fraction_bynode": 0.8321043377994284,
    "subsample": 0.6974385909748512,
    "colsample_bytree": 0.6568268046410831,
    "subsample_freq": 5,
    "drop_rate": 0.24668126335938073,
    "max_drop": 28,
    "skip_drop": 0.5591506516119614,
    "uniform_drop": True,
    "xgboost_dart_mode": True,
    "objective": "binary",
    "random_state": 13,
    "verbose": -1,
    "force_col_wise": True
}

doc2vec_param: Dict[str, Any] = {
    'vector_size': 150,
    'dm': 1,
    'window': 9,
    'min_count': 0,
    'alpha': 0.014996471116783728,
    'sample': 4.4713728630733355e-05,
    'epochs': 840,
    'negative': 12,
    'workers': 1,
    'seed': 13
}

# Create classifier
lightgbm_model = lgb.LGBMClassifier(**gbm_params)

# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    train_df = pickle.load(f)

input_path = "data/test_df2.pkl"
with open(input_path, "rb") as f:
    test_df = pickle.load(f)

fp4096_list, bit_list = fin(train_df, 3, 4096)
model = make_doc2vector(train_df, "description_gensim", bit_list, doc2vec_param)
X_train_vec, X_test_vec = make_fp2vector2(model, train_df, test_df)

fpdoc_results = main(train_df, test_df, X_train_vec, X_test_vec, lightgbm_model)


In [29]:
li = []
for category, result in fpdoc8192_results.items():
    print(f"## {category} ##")
    print(fpdoc8192_results[category]['f1']["test_scores"])
    li.append(fpdoc8192_results[category]['f1']["test_scores"])
print("")
print(np.mean(li))

## antioxidant ##
[0.6086956521739131]
## anti_inflammatory_agent ##
[0.6617647058823529]
## allergen ##
[0.6464646464646465]
## dye ##
[0.88]
## toxin ##
[0.625]
## flavouring_agent ##
[0.7619047619047619]
## agrochemical ##
[0.7962962962962963]
## volatile_oil ##
[0.9090909090909091]
## antibacterial_agent ##
[0.6530612244897959]
## insecticide ##
[0.7123287671232876]

0.7254606963425964


In [34]:
with open("result_unseen_prediction/fpdoc2vec.pkl", "rb") as f:
    a = pickle.load(f)
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in a.items():
    print(f"## {category} ##")
    print(a[category]['f1']["test_scores"])
    li.append(a[category]['f1']["test_scores"])
print("")
print(np.mean(li))

## antioxidant ##
[0.6595744680851063]
## anti_inflammatory_agent ##
[0.6507936507936508]
## allergen ##
[0.6530612244897959]
## dye ##
[0.9215686274509803]
## toxin ##
[0.5882352941176471]
## flavouring_agent ##
[0.7441860465116279]
## agrochemical ##
[0.7692307692307693]
## volatile_oil ##
[0.8163265306122449]
## antibacterial_agent ##
[0.6474820143884892]
## insecticide ##
[0.7397260273972602]

0.7190184653077573


In [16]:
with open("result_unseen_prediction/fpdoc2vec.pkl", "wb") as f:
    pickle.dump(fpdoc_results, f)

In [77]:
# {'f1': [], 'mcc': [], 'balanced_accuracy': [], 'roc_auc': [], 'kappa': [], 'pr_auc': []}

with open("result_unseen_prediction/descriptor.pkl", "rb") as f:
    a = pickle.load(f)
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in a.items():
    print(f"## {category} ##")
    print(a[category]['f1']["test_scores"])
    li.append(a[category]['f1']["test_scores"])
print("")
print(np.mean(li))

## antioxidant ##
[0.6296296296296297]
## anti_inflammatory_agent ##
[0.5822784810126582]
## allergen ##
[0.6379310344827587]
## dye ##
[0.8073394495412844]
## toxin ##
[0.711864406779661]
## flavouring_agent ##
[0.7755102040816326]
## agrochemical ##
[0.7433628318584071]
## volatile_oil ##
[0.7450980392156863]
## antibacterial_agent ##
[0.6545454545454545]
## insecticide ##
[0.6923076923076923]

0.6979867223454865


In [61]:
lr_params: Dict[str, Any] = {
       "C": 5.378351934170314, 
       "penalty": "l1", 
       "max_iter": 4300, 
       "class_weight": None, 
       "tol": 0.003446628082848086, 
       "solver": "liblinear", 
       "random_state": 0
}
# lr_params: Dict[str, Any] = {
#        "C": 0.001899363614004594, 
#        "penalty": "l2", 
#        "max_iter": 5000, 
#        "class_weight": None, 
#        "tol": 0.0002926347374178257, 
#        "solver": "newton-cg", 
#        "random_state": 0
# }

lr = LogisticRegression(**lr_params)
input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    train_df = pickle.load(f)

input_path = "data/test_df2.pkl"
with open(input_path, "rb") as f:
    test_df = pickle.load(f)

# FP Doc2Vec approach
X_train_vec, X_test_vec = make_fp2vector("train_df_doc2vec.model", train_df, test_df)
fpdoc_results = main(train_df, test_df, X_train_vec, X_test_vec, lr)

In [62]:
li = []
for category, result in fpdoc_results.items():
    print(f"## {category} ##")
    print(fpdoc_results[category]['f1']["test_scores"])
    li.append(fpdoc_results[category]['f1']["test_scores"])
print("")
print(np.mean(li))

## antioxidant ##
[0.55]
## anti_inflammatory_agent ##
[0.6153846153846154]
## allergen ##
[0.62]
## dye ##
[0.9158878504672897]
## toxin ##
[0.5333333333333333]
## flavouring_agent ##
[0.6976744186046512]
## agrochemical ##
[0.6923076923076923]
## volatile_oil ##
[0.782608695652174]
## antibacterial_agent ##
[0.6290322580645161]
## insecticide ##
[0.7164179104477612]

0.6752646774262033


!!! ECFP !!!

In [42]:
# FP Doc2Vec approach
X_train_vec, X_test_vec = fin(train_df, 3, 4096)[0], fin(test_df, 3, 4096)[0]
ecfp_results = main(train_df, test_df, X_train_vec, X_test_vec, lightgbm_model)


In [13]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in ecfp_results.items():
    print(f"## {category} ##")
    print(ecfp_results[category]['mcc']["test_scores"])
    li.append(ecfp_results[category]['mcc']["test_scores"])
print("")
print(np.mean(li))

## antioxidant ##
[0.5586839748200438]
## anti_inflammatory_agent ##
[0.5003558719509348]
## allergen ##
[0.4976382925428671]
## dye ##
[0.8720217469035048]
## toxin ##
[0.5208969232649949]
## flavouring_agent ##
[0.5999593186112382]
## agrochemical ##
[0.6550169343766138]
## volatile_oil ##
[0.6496412977928773]
## antibacterial_agent ##
[0.4721937120383588]
## insecticide ##
[0.5752905933545015]

0.5901698665655934


In [ ]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in ecfp_results.items():
    print(f"## {category} ##")
    print(ecfp_results[category]['mcc']["test_scores"])
    li.append(ecfp_results[category]['mcc']["test_scores"])
print("")
print(np.mean(li))

In [22]:
with open("result_unseen_prediction/ecfp.pkl", "wb") as f:
    pickle.dump(ecfp_results, f)

!!! Descriptor !!!

In [23]:
def make_descriptors(
    input_file: str, 
    train_df: pd.DataFrame, 
) -> Dict[str, Dict[str, float]]:

    # Load dataset
    with open(input_file, "rb") as f:
        df = pickle.load(f)
        
    # Split data into train and test sets
    train_df1 = df[df["inchikey"].isin(list(train_df["inchikey"]))]
    test_df1 = df.drop(train_df1.index)
    
    # Extract descriptor columns (from column 14 onward)
    train_desc = np.array(train_df1.select_dtypes(include=[np.floating]))
    test_desc = np.array(test_df1.select_dtypes(include=[np.floating]))

    return train_df1, test_df1, train_desc, test_desc

In [32]:
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

gbm_params: Dict[str, Any] = {
    "boosting_type": "dart",
    "num_leaves": 48,
    "max_depth": 5,
    "learning_rate": 0.04166324251391809,
    "n_estimators": 736,
    "class_weight": "balanced",
    "min_split_gain": 0.009346925180781129,
    "min_child_weight": 0.0007929549087822909,
    "min_child_samples": 37,
    "reg_alpha": 1.757104268180148,
    "reg_lambda": 1.463369722508726,
    "feature_fraction": 0.50163362868711,
    "feature_fraction_bynode": 0.8321043377994284,
    "subsample": 0.6974385909748512,
    "colsample_bytree": 0.6568268046410831,
    "subsample_freq": 5,
    "drop_rate": 0.24668126335938073,
    "max_drop": 28,
    "skip_drop": 0.5591506516119614,
    "uniform_drop": True,
    "xgboost_dart_mode": True,
    "objective": "binary",
    "random_state": 0,
    "verbose": -1,
    "force_col_wise": True
}

# Create classifier
lightgbm_model = lgb.LGBMClassifier(**gbm_params)

input_descriptor_path = "data/10genre_32descriptor2.pkl"
input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    data_df = pickle.load(f)
    
train_df, test_df ,X_train_vec, X_test_vec = make_descriptors(input_descriptor_path, data_df)
descriptor_results = main(train_df, test_df, X_train_vec, X_test_vec, lightgbm_model)

In [33]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in descriptor_results.items():
    print(f"## {category} ##")
    print(descriptor_results[category]['f1']["test_scores"])
    li.append(descriptor_results[category]['f1']["test_scores"])
print("")
print(np.mean(li))

## antioxidant ##
[0.6422018348623854]
## anti_inflammatory_agent ##
[0.5641025641025641]
## allergen ##
[0.6446280991735537]
## dye ##
[0.8256880733944955]
## toxin ##
[0.7666666666666667]
## flavouring_agent ##
[0.76]
## agrochemical ##
[0.7387387387387387]
## volatile_oil ##
[0.7450980392156863]
## antibacterial_agent ##
[0.654320987654321]
## insecticide ##
[0.7027027027027027]

0.7044147706511115


In [29]:
with open("result_unseen_prediction/descriptor.pkl", "wb") as f:
    pickle.dump(descriptor_results, f)

In [31]:
input_path = "data/10genre_32descriptor.pkl"
with open(input_path, "rb") as f:
    df = pickle.load(f)
df

,NAME,inchikey,smiles,ROMol,antioxidant,anti_inflammatory_agent,allergen,dye,toxin,flavouring_agent,...,SlogP_VSA11,SlogP_VSA12,SlogP_VSA2,TPSA,EState_VSA1,EState_VSA10,EState_VSA11,EState_VSA2,FractionCSP3,MolLogP
0,(-)-epicatechin,PFTAWBLQPZVEMU-UKRRQHHQSA-N,Oc1cc(O)c2c(c1)O[C@H](c1ccc(O)c(O)c1)[C@H](O)C2,<rdkit.Chem.rdchem.Mol object at 0x36c999080>,antioxidant,No,No,No,No,No,...,2.232141,-0.441701,-0.221602,0.065385,-0.316866,0.428217,-0.080350,0.499483,-0.748492,-0.211774
1,"2,6-dichlorobenzonitrile",YOYAIZYFCNQIRF-UHFFFAOYSA-N,N#Cc1c(Cl)cccc1Cl,<rdkit.Chem.rdchem.Mol object at 0x36f590fe0>,No,No,No,No,No,No,...,-0.507841,1.339734,-0.823300,-0.722586,-0.584523,-0.899389,-0.080350,-0.808066,-1.430527,0.148263
2,aspartame,IAOZJIPTCAWIRG-QWRGUYRKSA-N,COC(=O)[C@H](Cc1ccccc1)NC(=O)[C@@H](N)CC(=O)O,<rdkit.Chem.rdchem.Mol object at 0x36f591bc0>,No,No,No,No,No,flavouring_agent,...,-0.507841,-0.441701,-0.021727,0.141279,0.212453,-0.151493,-0.080350,-0.569342,-0.212607,-0.718727
3,azlocillin,JTWOMNBEOCYFNV-NFFDBFGFSA-N,CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](NC(=O)N3CCNC3=...,<rdkit.Chem.rdchem.Mol object at 0x3679d2480>,No,No,allergen,No,No,No,...,-0.507841,0.461374,0.699105,0.409092,0.689144,0.612626,-0.080350,-0.564734,0.104052,-0.608094
4,betulinic acid,QGJZLNKBHJESQX-FZFNOLFKSA-N,C=C(C)[C@@H]1CC[C@]2(C(=O)O)CC[C@]3(C)[C@H](CC...,<rdkit.Chem.rdchem.Mol object at 0x36f591b70>,No,anti_inflammatory_agent,No,No,No,No,...,-0.507841,-0.441701,-0.399435,-0.415552,-0.334924,-0.119048,-0.080350,0.444214,1.638630,1.301388
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3921,cytochalasin D,SDZRWUKZFQQKKV-FJUULPFHSA-N,C=C1[C@@H](O)C2/C=C/C[C@H](C)C(=O)[C@](C)(O)/C...,<rdkit.Chem.rdchem.Mol object at 0x35b6e76f0>,No,No,No,No,toxin,No,...,-0.507841,-0.441701,0.160434,0.088590,0.444784,0.379550,-0.080350,0.071240,0.274561,0.162463
3922,fenticlor,ANUSOIHIIPAHJV-UHFFFAOYSA-N,Oc1ccc(Cl)cc1Sc1cc(Cl)ccc1O,<rdkit.Chem.rdchem.Mol object at 0x35ba2f740>,No,No,allergen,No,No,No,...,0.588152,2.242809,-0.629057,-0.570889,-0.584523,-0.368347,-0.080350,-0.380536,-1.430527,0.609773
3923,6-methylcoumarin,FXFYOPQLGGEACP-UHFFFAOYSA-N,Cc1ccc2oc(=O)ccc2c1,<rdkit.Chem.rdchem.Mol object at 0x35ba2f790>,No,No,allergen,No,No,No,...,-0.507841,-0.441701,-0.823300,-0.664164,-0.584523,-0.650090,-0.080350,-0.598908,-1.089509,-0.060190
3924,"3,4,5-trimethoxycinnamic acid",YTFVRYKNXDADBI-UHFFFAOYSA-N,COc1cc(C=CC(=O)O)cc(OC)c1OC,<rdkit.Chem.rdchem.Mol object at 0x35b6e77e0>,No,No,allergen,No,No,No,...,1.136148,-0.441701,-0.206983,-0.347666,-0.453647,-0.650090,-0.080350,-0.808066,-0.577983,-0.139683


In [8]:
li=['MaxEStateIndex', 'MinEStateIndex', 'qed', 'MolWt', 'MaxPartialCharge', 
                    'MinPartialCharge', 'FpDensityMorgan1', 'FpDensityMorgan2', 'FpDensityMorgan3', 
                    'BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI', 
                    'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW', 'BalabanJ', 'BertzCT', 'Chi0', 
                    'Chi0n', 'Chi0v', 'Chi1', 'Chi1n', 'Chi1v', 'Chi2n', 'Chi2v', 'Chi3n', 'Chi3v', 
                    'Chi4n', 'Chi4v', 'HallKierAlpha', 'Kappa1', 'Kappa2', 'Kappa3', 'LabuteASA', 
                    'PEOE_VSA1', 'PEOE_VSA13', 'PEOE_VSA14', 'PEOE_VSA2', 'SMR_VSA1', 'SMR_VSA10', 
                    'SMR_VSA2', 'SMR_VSA9', 'SlogP_VSA1', 'SlogP_VSA11', 'SlogP_VSA12', 'SlogP_VSA2', 
                    'TPSA', 'EState_VSA1', 'EState_VSA10', 'EState_VSA11', 'EState_VSA2', 
                    'FractionCSP3', 'MolLogP', 'MolMR']
len(li)

56

In [12]:
input_descriptor_path = "data/10genre_79descriptor.pkl"
with open(input_descriptor_path, "rb") as f:
    df = pickle.load(f)
li = list(df.columns[15:])
len(li)

79

In [17]:
input_descriptor_path = "data/10genre_32descriptor.pkl"
with open(input_descriptor_path, "rb") as f:
    df = pickle.load(f)
li2 = list(df.columns[14:])
len(li2)

32

!!! xg boostでやってみる！！！

In [60]:
from xgboost import XGBClassifier as xgb
xgb_params: Dict[str, Any] = {
    "n_estimators": 810, 
    "max_depth": 9,
    "learning_rate": 0.0750274690444854, 
    "booster": "gbtree",
    "gamma": 0.06006649475004644, 
    "min_child_weight": 6.525033114236552,
    "max_delta_step": 4.3290483461345115, 
    "subsample": 0.6884228113550477,
    "colsample_bytree": 0.7444224692655531, 
    "reg_alpha": 1.7329549753275677,
    "reg_lambda": 4.33369477797301, 
    "scale_pos_weight": 9.91504160087765, 
    "random_state": 0,
    "n_jobs": 1,
    "eval_metric": 'logloss',
    "verbosity": 0
}

xg_boost = xgb(**xgb_params)
# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    train_df = pickle.load(f)

input_path = "data/test_df2.pkl"
with open(input_path, "rb") as f:
    test_df = pickle.load(f)

# FP Doc2Vec approach
X_train_vec, X_test_vec = fin(train_df, 3, 4096)[0], fin(test_df, 3, 4096)[0]
ecfpxg_results = main(train_df, test_df, X_train_vec, X_test_vec, xg_boost)

In [69]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in ecfpxg_results.items():
    print(f"## {category} ##")
    print(ecfpxg_results[category]['mcc']["test_scores"])
    li.append(ecfpxg_results[category]['mcc']["test_scores"])
print("")
print(np.mean(li))

## antioxidant ##
[0.5503765695893069]
## anti_inflammatory_agent ##
[0.5846271600785252]
## allergen ##
[0.5745286083979849]
## dye ##
[0.9054538327104661]
## toxin ##
[0.5874658299004295]
## flavouring_agent ##
[0.7367649614868775]
## agrochemical ##
[0.7516000680408319]
## volatile_oil ##
[0.7838685497621449]
## antibacterial_agent ##
[0.5846367228098723]
## insecticide ##
[0.6868752716210343]

0.6746197574397473


In [62]:
X_train_vec, X_test_vec = make_fp2vector("train_df_doc2vec.model", train_df, test_df)
fpdocxg_results = main(train_df, test_df, X_train_vec, X_test_vec, xg_boost)

In [70]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in fpdocxg_results.items():
    print(f"## {category} ##")
    print(fpdocxg_results[category]['mcc']["test_scores"])
    li.append(fpdocxg_results[category]['mcc']["test_scores"])
print("")
print(np.mean(li))

## antioxidant ##
[0.5578246964782108]
## anti_inflammatory_agent ##
[0.5893806990881459]
## allergen ##
[0.6255902338739395]
## dye ##
[0.9113467177983307]
## toxin ##
[0.598332032013941]
## flavouring_agent ##
[0.7191704661802111]
## agrochemical ##
[0.7340401533949921]
## volatile_oil ##
[0.8016190434022037]
## antibacterial_agent ##
[0.5847628866160777]
## insecticide ##
[0.6973614476320705]

0.6819428376478124


In [67]:
input_descriptor_path = "data/10genre_32descriptor.pkl"
train_df1, test_df1, train_desc, test_desc = make_descriptors(input_descriptor_path, test_df)
descriptorxg_results = main(train_df1, test_df1, train_desc, test_desc, xg_boost)

In [68]:
categories = [
        'antioxidant', 'anti_inflammatory_agent', 'allergen', 'dye', 'toxin', 
        'flavouring_agent', 'agrochemical', 'volatile_oil', 'antibacterial_agent', 'insecticide'
    ]
li = []
for category, result in __.items():
    print(f"## {category} ##")
    print(descriptorxg_results[category]['mcc']["test_scores"])
    li.append(descriptorxg_results[category]['mcc']["test_scores"])
print("")
print(np.mean(li))

## antioxidant ##
[0.421878823061788]
## anti_inflammatory_agent ##
[0.3513400291955054]
## allergen ##
[0.3401580555748893]
## dye ##
[0.746238696316604]
## toxin ##
[0.3132461787141121]
## flavouring_agent ##
[0.5046262374498579]
## agrochemical ##
[0.5742199341139289]
## volatile_oil ##
[0.6883528770738598]
## antibacterial_agent ##
[0.24701160319785692]
## insecticide ##
[0.47100128427116017]

0.4658073718969563


TOtoxin

fpdoc2vec

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

gbm_params: Dict[str, Any] = {
    "boosting_type": "gbdt",
    "num_leaves": 248,
    "max_depth": 12,
    "learning_rate": 0.17528053966450757,
    "n_estimators": 179,
    "class_weight": "balanced",
    "min_split_gain": 0.0348475566555026,
    "min_child_weight": 0.0004517270526861858,
    "min_child_samples": 37,
    "reg_alpha": 1.745112815468737,
    "reg_lambda": 0.479879372850994,
    "feature_fraction": 0.7463013314373087,
    "feature_fraction_bynode": 0.6545165345529773,
    "subsample": 0.5596715055348624,
    "colsample_bytree": 0.5068084541058617,
    "subsample_freq": 9,
    "drop_rate": 0.11197887287190003,
    "max_drop": 40,
    "skip_drop": 0.4197667702758505,
    "uniform_drop": False,
    "xgboost_dart_mode": True,
    "objective": "binary",
    "random_state": 0,
    "verbose": -1,
    "force_col_wise": True
}

# Create classifier
lightgbm_model = lgb.LGBMClassifier(**gbm_params)

# Load dataset for later use
# Example usage - replace with your actual file paths
input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    train_df = pickle.load(f)

input_path = "data/test_df2.pkl"
with open(input_path, "rb") as f:
    test_df = pickle.load(f)

# FP Doc2Vec approach
X_train_vec, X_test_vec = make_fp2vector("TOtoxin_fpdoc2vec.model", train_df, test_df)
fpdoc_results = main(train_df, test_df, X_train_vec, X_test_vec, lightgbm_model)


In [6]:
li = []
for category, result in fpdoc_results.items():
    print(f"## {category} ##")
    print(fpdoc_results[category]['f1']["test_scores"])
    li.append(fpdoc_results[category]['f1']["test_scores"])
print("")
print(np.mean(li))

## antioxidant ##
[0.5681818181818182]
## anti_inflammatory_agent ##
[0.6201550387596899]
## allergen ##
[0.6464646464646465]
## dye ##
[0.9142857142857143]
## toxin ##
[0.5454545454545454]
## flavouring_agent ##
[0.8095238095238095]
## agrochemical ##
[0.7326732673267327]
## volatile_oil ##
[0.8444444444444444]
## antibacterial_agent ##
[0.6666666666666666]
## insecticide ##
[0.7323943661971831]

0.7080244317305251


In [7]:
with open("result_unseen_prediction/TOtoxin_fpdoc2vec.pkl", "wb") as f:
    pickle.dump(fpdoc_results, f)

ECFP

In [8]:
# FP Doc2Vec approach
X_train_vec, X_test_vec = fin(train_df, 3, 4096)[0], fin(test_df, 3, 4096)[0]
ecfp_results = main(train_df, test_df, X_train_vec, X_test_vec, lightgbm_model)

In [10]:
li = []
for category, result in ecfp_results.items():
    print(f"## {category} ##")
    print(ecfp_results[category]['f1']["test_scores"])
    li.append(ecfp_results[category]['f1']["test_scores"])
print("")
print(np.mean(li))

## antioxidant ##
[0.5811965811965812]
## anti_inflammatory_agent ##
[0.6211180124223602]
## allergen ##
[0.6349206349206349]
## dye ##
[0.897196261682243]
## toxin ##
[0.5846153846153846]
## flavouring_agent ##
[0.6229508196721312]
## agrochemical ##
[0.6885245901639344]
## volatile_oil ##
[0.7017543859649122]
## antibacterial_agent ##
[0.611764705882353]
## insecticide ##
[0.6382978723404256]

0.658233924886096


In [13]:
with open("result_unseen_prediction/TOtoxin_ecfp.pkl", "wb") as f:
    pickle.dump(ecfp_results, f)

Descriptor

In [12]:
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

# Create classifier
lightgbm_model = lgb.LGBMClassifier(**gbm_params)

input_descriptor_path = "data/10genre_32descriptor.pkl"
input_path = "data/train_df2.pkl"
with open(input_path, "rb") as f:
    train_df = pickle.load(f)
train_df, test_df ,X_train_vec, X_test_vec = make_descriptors(input_descriptor_path, train_df)
descriptor_results = main(train_df, test_df, X_train_vec, X_test_vec, lightgbm_model)

In [14]:
li = []
for category, result in descriptor_results.items():
    print(f"## {category} ##")
    print(descriptor_results[category]['f1']["test_scores"])
    li.append(descriptor_results[category]['f1']["test_scores"])
print("")
print(np.mean(li))

## antioxidant ##
[0.631578947368421]
## anti_inflammatory_agent ##
[0.6428571428571429]
## allergen ##
[0.6548672566371682]
## dye ##
[0.8256880733944955]
## toxin ##
[0.7037037037037037]
## flavouring_agent ##
[0.723404255319149]
## agrochemical ##
[0.7636363636363637]
## volatile_oil ##
[0.7916666666666666]
## antibacterial_agent ##
[0.6538461538461539]
## insecticide ##
[0.684931506849315]

0.7076180070278579


In [15]:
with open("result_unseen_prediction/TOtoxin_descriptor.pkl", "wb") as f:
    pickle.dump(descriptor_results, f)